# Shor’s Algorithm for ECDLP-Based Private Key Recovery and Message Decryption

This notebook demonstrates the mathematical workflow of Shor's algorithm for solving the Elliptic Curve Discrete Logarithm Problem (ECDLP) on a deliberately small elliptic curve. After pasting a captured encrypted message with the recipient public key embedded in its payload, the notebook reconstructs the discrete logarithm recovery process and uses the recovered private scalar to derive the shared secret and decrypt the AES-GCM payload.

The demonstration proceeds in three stages:

- validate the captured message and user information;
- construct the hidden-relation function

$$
f(a,b)=aG+bQ,
$$

and simulate the Quantum Phase Estimation (QPE) and Quantum Fourier Transform (QFT) stages of Shor's algorithm;

- recover the private scalar from the simulated measurement relation, derive the shared ECDH secret, generate the AES-256 key, and decrypt the AES-GCM payload.

Rather than performing a classical brute-force search, the notebook follows the mathematical framework of Shor's algorithm. The quantum computation is represented through an educational simulation of the period-finding and phase-estimation process.

This notebook is intended solely for educational purposes. The elliptic curve parameters are intentionally small so that the decryptoin be possible. It does not demonstrate a practical decryption against standardized elliptic-curve cryptography, modern end-to-end encrypted messaging systems, HTTPS, or production cryptographic implementations.


# Part 1: Paste the captured payload

Paste either the encrypted message object from the backend/app output or the full Socket.IO frame from Wireshark into `MESSAGE_JSON`, for example `42["message_created", {...}]`.

If wanted to paste the raw Wireshark frame into its own cell, run the imports cell first, then create a new cell that starts with `socketio_message` and paste the raw frame on the next line.

Set `PAYLOAD_NUMBER` to choose which encrypted payload inside the message should be decrypted. The selected payload must include `recipientPublicKey` so the captured packet is self-contained.

In [ ]:
MESSAGE_JSON = r"""

"""
PAYLOAD_NUMBER = 1

## Imports and environment checks

The notebook uses Qiskit for the small Fourier/QPE circuit illustrations and Python arithmetic for the  educational elliptic-curve simulation. AES-GCM decryption uses `pycryptodome`, matching the existing project notebook.

In [ ]:
import base64
import hashlib
import json
import math
import os
import random
from collections import defaultdict

os.environ.setdefault("MPLCONFIGDIR", os.path.abspath(".matplotlib-cache"))

try:
    from qiskit import QuantumCircuit, transpile
    from qiskit_aer import AerSimulator
    from qiskit.visualization import plot_histogram
    QISKIT_AVAILABLE = True
except Exception as exc:
    QISKIT_AVAILABLE = False
    QISKIT_IMPORT_ERROR = exc

try:
    from IPython.display import display
    from IPython.core.magic import register_cell_magic
except Exception:
    display = None
    register_cell_magic = None

if register_cell_magic is not None:
    @register_cell_magic
    def socketio_message(_line, cell):
        global MESSAGE_JSON
        MESSAGE_JSON = cell.strip()
        print("Stored Socket.IO frame in MESSAGE_JSON. Now run the parse/select cell below.")

print("Qiskit available:", QISKIT_AVAILABLE)
if not QISKIT_AVAILABLE:
    print("Qiskit import issue:", QISKIT_IMPORT_ERROR)

## Parse and select the payload

This cell unwraps Socket.IO frames when needed, then checks that the pasted message has the fields required for the demonstration.

In [ ]:
def parse_json_block(raw, label):
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{label} is not valid JSON: {exc}") from exc

def parse_message_block(raw):
    raw = raw.strip()
    if not raw:
        raise ValueError("MESSAGE_JSON is empty")

    first_json_index = min(
        (index for index in (raw.find("{"), raw.find("[")) if index != -1),
        default=-1,
    )
    if first_json_index > 0 and raw[:first_json_index].isdigit():
        raw = raw[first_json_index:]

    parsed = parse_json_block(raw, "MESSAGE_JSON")

    if isinstance(parsed, dict):
        return parsed

    if isinstance(parsed, list):
        if len(parsed) >= 2 and isinstance(parsed[0], str) and isinstance(parsed[1], dict):
            return parsed[1]
        if len(parsed) == 1 and isinstance(parsed[0], dict) and isinstance(parsed[0].get("message"), dict):
            return parsed[0]["message"]

    raise ValueError(
        "MESSAGE_JSON must be a message object, a Socket.IO event frame, or an ack frame containing a message"
    )

message = parse_message_block(MESSAGE_JSON)
assert "senderPublicKey" in message, "MESSAGE_JSON must include senderPublicKey"
assert "payloads" in message and message["payloads"], "MESSAGE_JSON must include at least one payload"
assert 1 <= PAYLOAD_NUMBER <= len(message["payloads"]), "PAYLOAD_NUMBER is outside the available payload range"

payload = message["payloads"][PAYLOAD_NUMBER - 1]
recipient_id = payload["recipientId"]
assert "recipientPublicKey" in payload, "Selected payload must include recipientPublicKey"

print("Selected payload number:", PAYLOAD_NUMBER)
print("Recipient id:", recipient_id)
print("Sender public key:", message["senderPublicKey"])
print("Recipient public key:", payload["recipientPublicKey"])

# Part 2 — Shor/ECDLP quantum recovery workflow

## Define the curve and the hidden-relation function

The chat demo uses a deliberately tiny elliptic curve:

$$
y^2 \equiv x^3+2x+3 \pmod{211}.
$$

The generator is

$$
G=(199,192),
$$

with subgroup order

$$
r=204.
$$

For a public key $Q=dG$, Shor's elliptic-curve discrete-log algorithm uses the function

$$
f(a,b)=aG+bQ.
$$

Because $Q=dG$, this is secretly

$$
f(a,b)=(a+bd)G.
$$

The private scalar $d$ is encoded implicitly through the hidden relation and is recovered from the Fourier measurement followed by classical modular arithmetic.

In [ ]:
DEMO_CURVE = {
    "name": "EMRAN AZEEZI",
    "p": 211,
    "a": 2,
    "b": 3,
    "G": (199, 192),
    "order": 204,
}

p = DEMO_CURVE["p"]
a_curve = DEMO_CURVE["a"]
b_curve = DEMO_CURVE["b"]
G = DEMO_CURVE["G"]
r = DEMO_CURVE["order"]
POINT_AT_INFINITY = None

def mod(value):
    return value % p

def inverse_mod(value, modulus=p):
    return pow(value % modulus, -1, modulus)

def on_curve(P):
    if P is None:
        return True
    x, y = P
    return mod(y * y) == mod(x * x * x + a_curve * x + b_curve)

def point_add(P, R):
    if P is None:
        return R
    if R is None:
        return P

    x1, y1 = P
    x2, y2 = R

    if x1 == x2 and mod(y1 + y2) == 0:
        return None

    if P == R:
        numerator = 3 * x1 * x1 + a_curve
        denominator = 2 * y1
    else:
        numerator = y2 - y1
        denominator = x2 - x1

    slope = mod(numerator * inverse_mod(denominator, p))
    x3 = mod(slope * slope - x1 - x2)
    y3 = mod(slope * (x1 - x3) - y1)
    return (x3, y3)

def scalar_multiply(k, P=G):
    k %= r
    result = None
    addend = P
    while k:
        if k & 1:
            result = point_add(result, addend)
        addend = point_add(addend, addend)
        k >>= 1
    return result

def point_from_json(obj):
    return (int(obj["x"]), int(obj["y"]))

def point_to_json(P):
    if P is None:
        return None
    return {"x": P[0], "y": P[1]}

def shor_function(a_value, b_value, public_key):
    return point_add(scalar_multiply(a_value, G), scalar_multiply(b_value, public_key))

sender_public_key = point_from_json(message["senderPublicKey"])
recipient_public_key = point_from_json(payload["recipientPublicKey"])

assert on_curve(G), "Generator is not on the curve"
assert scalar_multiply(r, G) is None, "Configured subgroup order is incorrect"
assert on_curve(sender_public_key), "Sender public key is not on the curve"
assert on_curve(recipient_public_key), "Recipient public key is not on the curve"

print("Curve:", f"y^2 = x^3 + {a_curve}x + {b_curve} mod {p}")
print("Generator G:", G)
print("Subgroup order r:", r)
print("Recipient public key Q:", recipient_public_key)
print("Example f(2,3)=2G+3Q:", shor_function(2, 3, recipient_public_key))


## Shor/ECC quantum simulation 

This step demonstrates the quantum stage of Shor's algorithm for the elliptic-curve discrete logarithm problem.

The workflow proceeds through the following stages:

1. Construct the QPE control and worker registers.
2. Create a uniform superposition over the control registers.
3. Evaluate the oracle $f(a,b)=aG+bQ$ through a simulated reversible unitary.
4. Encode the hidden relation into quantum phases via phase kickback.
5. Apply the inverse Quantum Fourier Transform to convert phase information into measurable register values.
6. Simulate quantum measurement to obtain a Fourier sample.
7. Perform classical modular post-processing to recover the recipient's private scalar.
8. Verify the recovered scalar by checking $Q=dG$.

The oracle, phase evolution, Fourier interference, and quantum measurement are simulated mathematically for the educational elliptic-curve example used in this notebook.


In [ ]:
def qft_matrix(size, inverse=False):
    import cmath
    sign = -1 if inverse else 1
    scale = 1 / math.sqrt(size)
    return [
        [scale * cmath.exp(sign * 2j * math.pi * row * col / size) for col in range(size)]
        for row in range(size)
    ]

def build_function_fibers(public_key):
    fibers = defaultdict(list)
    for a_value in range(r):
        aG = scalar_multiply(a_value, G)
        for b_value in range(r):
            value = point_add(aG, scalar_multiply(b_value, public_key))
            fibers[value].append((a_value, b_value))
    return fibers

def fourier_distribution_for_fiber(fiber):
    import cmath
    probabilities = []
    norm = len(fiber) * r * r

    for u in range(r):
        for v in range(r):
            amplitude = 0j
            for a_value, b_value in fiber:
                phase = 2j * math.pi * (a_value * u + b_value * v) / r
                amplitude += cmath.exp(phase)

            probability = (abs(amplitude) ** 2) / norm
            if probability > 1e-10:
                probabilities.append((u, v, probability))

    total = sum(probability for _, _, probability in probabilities)
    return [(u, v, probability / total) for u, v, probability in probabilities]

def sample_frequency_pair(distribution):
    threshold = random.random()
    cumulative = 0.0
    for item in distribution:
        cumulative += item[2]
        if cumulative >= threshold:
            return item
    return distribution[-1]

def shor_ecc_measurement_simulation(public_key, max_attempts=50, seed=7):
    previous_state = random.getstate()
    random.seed(seed)

    try:
        print("QPE registers: |a>|b>|Y>")
        print("Superposition: uniform over a,b in Z_204")

        if QISKIT_AVAILABLE:
            register_demo = QuantumCircuit(4, 4)
            register_demo.h([0, 1, 2, 3])
            register_demo.measure([0, 1, 2, 3], [0, 1, 2, 3])
            simulator = AerSimulator()
            counts = simulator.run(transpile(register_demo, simulator), shots=1024).result().get_counts()
            print("Small Qiskit register demo counts:", counts)
        else:
            print("Small Qiskit register demo skipped: Qiskit is not installed in this environment.")

        print("Oracle unitary: |a>|b>|Y> -> |a>|b>|Y + aG + bQ>")
        fibers = build_function_fibers(public_key)

        measured_worker_point = random.choice(list(fibers.keys()))
        measured_fiber = fibers[measured_worker_point]
        print("Measured worker register:", measured_worker_point)
        print("Remaining collision-line states:", len(measured_fiber))

        print("Inverse Fourier step: 4 by 4 matrix sample")
        for row in qft_matrix(4, inverse=True):
            print([complex(round(value.real, 3), round(value.imag, 3)) for value in row])

        distribution = fourier_distribution_for_fiber(measured_fiber)
        print("Fourier measurement attempts:")

        attempts = []
        for attempt in range(1, max_attempts + 1):
            u, v, probability = sample_frequency_pair(distribution)
            gcd_value = math.gcd(u, r)
            record = {
                "attempt": attempt,
                "u": u,
                "v": v,
                "gcd(u,204)": gcd_value,
                "probability": probability,
            }

            if u != 0 and gcd_value == 1:
                recovered = (v * pow(u, -1, r)) % r
                record["recovered_d"] = recovered
                attempts.append(record)
                print(record)
                return recovered, measured_worker_point, measured_fiber, attempts

            attempts.append(record)
            print(record)

        raise RuntimeError("No invertible Fourier sample found. Increase max_attempts.")
    finally:
        random.setstate(previous_state)

recipient_private_key, measured_worker_point, measured_fiber, shor_attempts = shor_ecc_measurement_simulation(recipient_public_key)

print("Recovered recipient private scalar d:", recipient_private_key)
print("Verification dG == Q:", scalar_multiply(recipient_private_key, G) == recipient_public_key)


## Part 3 — Classical Post-Measurement Computation and Payload Decryption

After the quantum measurement stage, the notebook has recovered the recipient's private scalar $d_R$ corresponding to the selected public key

$$
Q_R = d_R G.
$$

At this point, the quantum computation is complete. The remaining steps are entirely classical.

The encrypted payload was generated using an ECDH-style key exchange. The sender computes the shared secret

$$
S = d_S Q_R,
$$

where $d_S$ is the sender's private scalar.

Using the recovered private scalar, the notebook computes the identical shared secret

$$
S = d_R Q_S,
$$

where $Q_S$ is the sender's public key.

Since

$$
d_SQ_R
=
d_S(d_RG)
=
d_R(d_SG)
=
d_RQ_S,
$$

both computations produce the same elliptic-curve point $S$.

The notebook then reconstructs this shared secret, derives the AES-256 key using the same key-derivation procedure as the chat application, and decrypts the selected AES-GCM payload.

In [ ]:
def derive_frontend_aes_key(shared_secret):
    """Match frontend/src/cryptoDemo.js deriveAesKeyBytes."""
    kdf_input = json.dumps(
        {
            "purpose": "quantum-chat-aes-gcm",
            "curve": DEMO_CURVE["name"],
            "sharedSecret": point_to_json(shared_secret),
        },
        separators=(",", ":"),
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.sha256(kdf_input).digest()

def validate_payload_bytes(payload):
    if "PASTE_" in payload.get("iv", "") or "PASTE_" in payload.get("ciphertext", ""):
        raise ValueError("Paste a real IV and ciphertext into MESSAGE_JSON before running decryption.")

    try:
        iv = base64.b64decode(payload["iv"], validate=True)
        ciphertext_with_tag = base64.b64decode(payload["ciphertext"], validate=True)
    except Exception as exc:
        raise ValueError("Payload IV/ciphertext must be valid base64 strings copied from the app output.") from exc

    if len(iv) != 12:
        raise ValueError(f"AES-GCM IV should be 12 bytes from the frontend, got {len(iv)} bytes.")
    if len(ciphertext_with_tag) <= 16:
        raise ValueError("Ciphertext is too short. WebCrypto AES-GCM ciphertext includes a 16-byte auth tag at the end.")

    return iv, ciphertext_with_tag

def aes_gcm_decrypt(aes_key, iv, ciphertext_with_tag):
    try:
        from Crypto.Cipher import AES
        ciphertext = ciphertext_with_tag[:-16]
        tag = ciphertext_with_tag[-16:]
        cipher = AES.new(aes_key, AES.MODE_GCM, nonce=iv)
        return cipher.decrypt_and_verify(ciphertext, tag)
    except ImportError:
        try:
            from cryptography.hazmat.primitives.ciphers.aead import AESGCM # type: ignore
            return AESGCM(aes_key).decrypt(iv, ciphertext_with_tag, None)
        except ImportError as exc:
            raise ImportError(
                "AES-GCM decryption needs pycryptodome or cryptography. "
                "Install one of them, for example: pip install pycryptodome"
            ) from exc

def decrypt_payload(payload, recipient_private_key, sender_public_key):
    shared_secret = scalar_multiply(recipient_private_key, sender_public_key)
    if shared_secret is None:
        raise ValueError("Shared secret is point-at-infinity; choose another payload.")

    aes_key = derive_frontend_aes_key(shared_secret)
    iv, ciphertext_with_tag = validate_payload_bytes(payload)
    try:
        plaintext = aes_gcm_decrypt(aes_key, iv, ciphertext_with_tag).decode("utf-8")
    except Exception as exc:
        raise ValueError(
            "AES-GCM authentication failed. This means the pasted payload does not match the recovered key. "
            "Check that MESSAGE_JSON, PAYLOAD_NUMBER, senderPublicKey, recipientPublicKey, IV, and ciphertext "
            "all come from the same captured message."
        ) from exc
    return plaintext, shared_secret, aes_key

def show_decrypted_content(plaintext):
    print("Decrypted plaintext:")
    print(plaintext)

try:
    plaintext, shared_secret, aes_key = decrypt_payload(payload, recipient_private_key, sender_public_key)
    print("Recovered recipient private key:", recipient_private_key)
    print("Shared secret:", point_to_json(shared_secret))
    print("Derived AES key hex:", aes_key.hex())
    show_decrypted_content(plaintext)
except ValueError as exc:
    print("Decryption not completed:", exc)


## References

- IBM Quantum. [Shor's algorithm tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/shors-algorithm).
